# Phase 4 Stage 1 — v4 Resume (QLoRA Qwen3.5-4B-Base)

**Muc dich:** Resume tu v3 epoch 2 ckpt (`weights/v3_adapter/checkpoint-2680`), train them 2 epoch tren cung data (`items_prompts_tv_3` 85K) de isolate tac dong cua technique:
- NEFTune alpha=3 (warm start)
- LR=5e-5 (mem hon vi adapter da warm)
- group_by_length=True (-10-15% time)
- Best ckpt theo eval generative RMSLE (KHONG theo eval CE — v3 da chung minh CE-RMSLE divergence)
- EarlyStoppingCallback patience=3

**Constraint khi resume LoRA adapter:** KHONG the doi `r`, `alpha`, `target_modules`, `dropout`. Chi doi: lr/scheduler/data/epochs/NEFTune/group_by_length.

**Reference:** `04_train_v3.ipynb` — moi pattern (manual collator, conv1d bf16 cast, PRICE_PREFIX masking) ke thua tu day.

| Param | v3 (proven) | v4-resume |
|-------|------------|-----------|
| Train size | 85,727 | 85,727 (same) |
| LoRA r / alpha / drop | 64 / 128 / 0.1 | 64 / 128 / 0.1 (frozen by adapter) |
| Target modules | 7 | 7 (frozen) |
| Epochs | 3 | **+2** (continue) |
| LR | 2e-4 | **5e-5** |
| Warmup ratio | 0.03 | **0.01** |
| NEFTune alpha | 0 | **3** |
| group_by_length | False | **True** |
| Eval strategy | steps (CE) | **steps + custom RMSLE 500 val** |
| Best by | last epoch | **eval_rmsle** |
| Eff batch | 64 | 64 |

**Ky vong:** RMSLE 0.40-0.42 (v3 final = 0.4426). Time ~10h tren 1x 3090Ti.

In [ ]:
# Chay neu chua co trong env:
#!uv add "transformers>=5.2.0" peft trl bitsandbytes accelerate datasets python-dotenv

In [ ]:
import os
import re
import sys
import gc
import json
import time
import math
import random
import numpy as np
from tqdm import tqdm
from pathlib import Path

import torch
import bitsandbytes as bnb
import torch.nn as nn
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
from peft import PeftModel, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
)
from trl import SFTTrainer, SFTConfig

from dataclasses import dataclass
from typing import Any, Dict, List
from transformers import PreTrainedTokenizerBase

@dataclass
class DataCollatorForCompletionOnlyLM:
    """Manual impl: trl.DataCollatorForCompletionOnlyLM removed in TRL 0.24.0 (giu nguyen tu v3)."""
    response_template: List[int]
    tokenizer: PreTrainedTokenizerBase
    ignore_index: int = -100

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_ids_list = [torch.tensor(f["input_ids"], dtype=torch.long) for f in features]
        max_len = max(len(x) for x in input_ids_list)
        bs = len(input_ids_list)

        padded = torch.full((bs, max_len), self.tokenizer.pad_token_id, dtype=torch.long)
        attn   = torch.zeros((bs, max_len), dtype=torch.long)
        labels = torch.full((bs, max_len), self.ignore_index, dtype=torch.long)

        tpl, tpl_len = self.response_template, len(self.response_template)

        for i, ids in enumerate(input_ids_list):
            n = len(ids)
            padded[i, :n] = ids
            attn[i, :n]   = 1
            for j in range(n - tpl_len, -1, -1):
                if ids[j : j + tpl_len].tolist() == tpl:
                    labels[i, j + tpl_len : n] = ids[j + tpl_len : n]
                    break

        return {"input_ids": padded, "attention_mask": attn, "labels": labels}

print("DataCollatorForCompletionOnlyLM: manual impl OK")

NOTEBOOK_DIR = Path("__file__").parent if "__file__" in dir() else Path(".")
sys.path.insert(0, str(NOTEBOOK_DIR))
from utils.evaluator import compute_metrics, plot_predictions
from utils.rmsle_callback import RMSLEEvalCallback

print("Imports OK")
import transformers, peft, trl
for pkg, mod in [("torch", torch), ("transformers", transformers), ("peft", peft), ("trl", trl)]:
    print(f"  {pkg:<14}: {mod.__version__}")

In [ ]:
# --- v4-resume constants (Stage 1 plan_day5.md Section 4.2) ---

BASE_MODEL   = "Qwen/Qwen3.5-4B-Base"
DATASET_NAME = "SeanSunny/items_prompts_tv_3"

# Sequence (giu y nhu v3)
MAX_SEQ_LENGTH  = 192
MAX_NEW_TOKENS  = 4
QUESTION_PREFIX = "S\u1ea3n ph\u1ea9m n\u00e0y c\u00f3 gi\u00e1 bao nhi\u00eau ?\n"
PRICE_PREFIX    = "\n\nGi\u00e1 l\u00e0: "

# Resume — frozen by adapter (cannot change r/alpha/modules/dropout)
RESUME_ADAPTER  = "weights/v3_adapter/checkpoint-2680"   # epoch 2 cua v3
LORA_R              = 64    # echo-back de log
LORA_ALPHA          = 128
LORA_DROPOUT        = 0.1
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"]

# Training (v4-resume)
TRAIN_SIZE        = None         # None = full 85,727
VAL_CALLBACK_SIZE = 500          # subset cho RMSLE callback trong train
VAL_FULL_SIZE     = None         # None = full 3,926 val cho final eval
PLOT_SIZE         = 200          # so item ve scatter sau cung
NUM_EPOCHS        = 2            # them 2 ep tren v3 epoch 2
PER_DEVICE_BATCH  = 16
GRAD_ACCUM        = 4            # eff_batch = 64
LEARNING_RATE     = 5e-5         # giam tu 2e-4 (warm start)
LR_SCHEDULER      = "cosine"
WARMUP_RATIO      = 0.01         # ngan hon vi adapter da warm
WEIGHT_DECAY      = 0.001        # giu nhu v3
OPTIM             = "paged_adamw_32bit"
GRADIENT_CHECKPOINTING = True
GROUP_BY_LENGTH   = True         # bat o v4 (v3 = False)
NEFTUNE_ALPHA     = 3            # warm-start NEFTune
EVAL_STEPS        = 500          # eval gen RMSLE moi 500 step
SAVE_STEPS        = 500          # save aligned voi eval de load_best_model_at_end work
EARLY_STOP_PATIENCE = 3
LOGGING_STEPS     = 50
SEED              = 42

# Inference safety
PRED_CLAMP_MIN = 5
PRED_CLAMP_MAX = 1000
PARSE_REGEX    = r"[-+]?\d*\.\d+|\d+"

# Paths
ADAPTER_DIR     = NOTEBOOK_DIR / "weights" / "v4_resume_adapter"
RESULTS_FILE    = NOTEBOOK_DIR / "results" / "v4_resume_results.json"
HF_REPO_ADAPTER = "SeanSunny/qwen3.5-4b-vn-pricer-v4-resume"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"BASE_MODEL      : {BASE_MODEL}")
print(f"DATASET_NAME    : {DATASET_NAME}")
print(f"RESUME_ADAPTER  : {RESUME_ADAPTER}")
print(f"Training        : full data, +{NUM_EPOCHS} ep, eff_batch={PER_DEVICE_BATCH*GRAD_ACCUM}, lr={LEARNING_RATE}")
print(f"NEFTune alpha   : {NEFTUNE_ALPHA}")
print(f"group_by_length : {GROUP_BY_LENGTH}")
print(f"Best metric     : eval_rmsle (custom callback, n={VAL_CALLBACK_SIZE})")
print(f"ADAPTER_DIR     : {ADAPTER_DIR}")
print(f"RESULTS_FILE    : {RESULTS_FILE}")

In [ ]:
# GPU + HF login + dirs
assert torch.cuda.is_available(), "GPU khong kha dung."
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
cap = torch.cuda.get_device_capability()
print(f"bf16 : {'yes' if cap[0] >= 8 else 'no'} (compute {cap})")

env_path = NOTEBOOK_DIR.parent / ".env"
load_dotenv(env_path)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(HF_TOKEN)
    print(f"HF login OK (from {env_path})")
else:
    print(f"HF_TOKEN not set in {env_path}")
    login()

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
(NOTEBOOK_DIR / "results").mkdir(exist_ok=True)

resume_path = NOTEBOOK_DIR / RESUME_ADAPTER
assert resume_path.exists(), (
    f"RESUME_ADAPTER not found: {resume_path}. "
    "Anh download v3 epoch 2 tu Google Drive vao path nay truoc khi chay."
)
print(f"Resume ckpt OK : {resume_path}")
print("Dirs ready: weights/v4_resume_adapter/, results/")

## 1. Load base model (4-bit) + RESUME v3 adapter

**BAT BUOC `dtype=torch.bfloat16`** trong `from_pretrained` — neu thieu se crash conv1d cua Qwen3.5 GatedDeltaNet luc inference (lesson tu v1).

**Resume pattern:** `prepare_model_for_kbit_training` TRUOC, sau do `PeftModel.from_pretrained(..., is_trainable=True)` — KHONG goi `get_peft_model` vi adapter da configured.

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
if tokenizer.eos_token_id is None:
    tokenizer.eos_token = "<|endoftext|>"
    print("WARNING: set eos_token manually to <|endoftext|>")
print(f"EOS token : {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"PAD token : {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
base_model = prepare_model_for_kbit_training(
    base_model, use_gradient_checkpointing=GRADIENT_CHECKPOINTING
)
for _m in base_model.modules():
    if isinstance(_m, torch.nn.Conv1d):
        _m.to(torch.bfloat16)

# Resume adapter — is_trainable=True de tiep tuc fine-tune
model = PeftModel.from_pretrained(
    base_model, str(resume_path), is_trainable=True
)
model.print_trainable_parameters()
model.enable_input_require_grads()
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## 2. Dataset + preprocess + tokenize

Dung lai pipeline cua v3: cat summary token-level tu duoi, append PRICE_PREFIX + completion + EOS.

In [ ]:
ds = load_dataset(DATASET_NAME)
print(f"Train: {len(ds['train']):,} | Val: {len(ds['val']):,} | Test: {len(ds['test']):,}")

q_ids = tokenizer.encode(QUESTION_PREFIX, add_special_tokens=False)
p_ids = tokenizer.encode(PRICE_PREFIX, add_special_tokens=False)
TOKENS_FIXED = len(q_ids) + len(p_ids)
MAX_SUMMARY_TOKENS = MAX_SEQ_LENGTH - TOKENS_FIXED - MAX_NEW_TOKENS - 1 - 2
print(f"TOKENS_FIXED        : {TOKENS_FIXED}")
print(f"MAX_SUMMARY_TOKENS  : {MAX_SUMMARY_TOKENS}")

train_full_raw = ds["train"].shuffle(seed=SEED)
if TRAIN_SIZE is not None:
    train_full_raw = train_full_raw.select(range(TRAIN_SIZE))

val_full_raw = ds["val"]                                        # giu order goc cho final eval
val_callback_raw = ds["val"].shuffle(seed=SEED).select(range(VAL_CALLBACK_SIZE))

def preprocess(example):
    p = example["prompt"]
    summary = p[len(QUESTION_PREFIX):-len(PRICE_PREFIX)]
    summary_ids = tokenizer.encode(summary, add_special_tokens=False)
    if len(summary_ids) > MAX_SUMMARY_TOKENS:
        summary_ids = summary_ids[:MAX_SUMMARY_TOKENS]
        summary = tokenizer.decode(summary_ids, skip_special_tokens=True).rstrip()
    full_text = QUESTION_PREFIX + summary + PRICE_PREFIX + example["completion"] + "\n" + tokenizer.eos_token
    return {"text": full_text}

train_ds = train_full_raw.map(preprocess, desc="Preprocess train")
# val cho Trainer in-train CE eval (ngan, dung de tracking + early stop fallback)
val_ds_for_trainer = val_callback_raw.map(preprocess, desc="Preprocess val (trainer)")
print(f"train_ds: {len(train_ds):,} | val_ds_for_trainer: {len(val_ds_for_trainer):,}")

for i in range(2):
    t = train_ds[i]["text"]
    print(f"--- Sample {i} ---")
    print(repr(t[:200]))
    assert PRICE_PREFIX in t, f"ERROR: PRICE_PREFIX missing in sample {i}"
print("PRICE_PREFIX present: OK")

In [ ]:
def tokenize_fn(example):
    return tokenizer(example["text"], truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)

train_ds_tok = train_ds.map(tokenize_fn, batched=False, remove_columns=["text"])
val_ds_tok   = val_ds_for_trainer.map(tokenize_fn, batched=False, remove_columns=["text"])
print(f"Columns      : {train_ds_tok.column_names}")
print(f"Sample len   : {len(train_ds_tok[0]['input_ids'])}")

# Them cot 'length' cho group_by_length (Trainer can column nay khi GROUP_BY_LENGTH=True)
train_ds_tok = train_ds_tok.map(
    lambda ex: {"length": len(ex["input_ids"])}, desc="Compute length"
)
print(f"Has length col: {'length' in train_ds_tok.column_names}")

## 3. DataCollator + verify mask + RMSLE callback subset

Mask verify fail-loud (R6 v3). RMSLE callback nhan `val_callback_raw` (500 items co dinh, seed=42).

In [ ]:
response_template_ids = tokenizer.encode(PRICE_PREFIX, add_special_tokens=False)
decoded_back = tokenizer.decode(response_template_ids)
print(f"response_template_ids : {response_template_ids}")
print(f"decoded back          : {decoded_back!r}")
assert decoded_back == PRICE_PREFIX, "PRICE_PREFIX decode mismatch — abort."

collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,
    tokenizer=tokenizer,
)

# Mask verify fail-loud
sample_text = train_ds[0]["text"]
tokenized = tokenizer(sample_text, return_tensors="pt", max_length=MAX_SEQ_LENGTH, truncation=True)
batch_in = [{
    "input_ids": tokenized["input_ids"][0].tolist(),
    "attention_mask": tokenized["attention_mask"][0].tolist(),
}]
labels = collator(batch_in)["labels"][0]
non_masked = labels[labels != -100]
if len(non_masked) == 0:
    raise RuntimeError("Mask verify FAILED: response_template_ids not found.")
print(f"Decoded non-masked    : {tokenizer.decode(non_masked.tolist(), skip_special_tokens=False)!r}")
print(f"Sample completion     : {train_full_raw[0]['completion']!r}")
print("Mask verify PASS.")

# RMSLE callback — eval gen RMSLE tren 500 val co dinh moi EVAL_STEPS
rmsle_callback = RMSLEEvalCallback(
    tokenizer=tokenizer,
    val_subset=val_callback_raw,
    max_new_tokens=MAX_NEW_TOKENS,
    clamp_min=PRED_CLAMP_MIN,
    clamp_max=PRED_CLAMP_MAX,
)
print(f"RMSLEEvalCallback ready (n={len(val_callback_raw)} items).")

## 4. Train — resume + NEFTune + group_by_length + best by RMSLE

- `eval_strategy="steps"` + `save_strategy="steps"` + `eval_steps == save_steps` (bat buoc cho `load_best_model_at_end`).
- `metric_for_best_model="eval_rmsle"`, `greater_is_better=False`.
- `EarlyStoppingCallback(patience=3)` — auto stop khi eval_rmsle khong improve 3 lan lien tiep.
- `neftune_noise_alpha=3` — warm start NEFTune (nhe hon alpha=5 cho scratch).
- `group_by_length=True` — giam padding waste.

In [ ]:
torch.cuda.reset_peak_memory_stats()
t_train_start = time.time()

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds_tok,
    eval_dataset=val_ds_tok,
    data_collator=collator,
    callbacks=[
        rmsle_callback,
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE),
    ],
    args=SFTConfig(
        output_dir=str(ADAPTER_DIR),
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULER,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        optim=OPTIM,
        bf16=True,
        max_grad_norm=0.3,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        train_sampling_strategy="group_by_length" if GROUP_BY_LENGTH else "random",
        length_column_name="length",
        neftune_noise_alpha=NEFTUNE_ALPHA,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_rmsle",
        greater_is_better=False,
        logging_steps=LOGGING_STEPS,
        report_to="none",
        seed=SEED,
    ),
)
trainer.train()
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

total_train_sec = time.time() - t_train_start
final_vram_peak = torch.cuda.max_memory_allocated() / 1e9
print(f"Training complete: {total_train_sec/60:.1f} min ({total_train_sec/3600:.1f} hr) | VRAM peak: {final_vram_peak:.2f} GB")

log_history = trainer.state.log_history
train_losses = [(int(e["step"]), float(e["loss"])) for e in log_history if "loss" in e and "eval_loss" not in e]
eval_losses  = [(int(e["step"]), float(e["eval_loss"])) for e in log_history if "eval_loss" in e]
eval_rmsle_curve = [
    (int(e["step"]), float(e["eval_rmsle"])) for e in log_history if "eval_rmsle" in e
]
print(f"Train loss entries: {len(train_losses)} | Eval CE: {len(eval_losses)} | Eval RMSLE: {len(eval_rmsle_curve)}")
if eval_rmsle_curve:
    best_step, best_rmsle_subset = min(eval_rmsle_curve, key=lambda x: x[1])
    print(f"Best eval_rmsle (500 val subset): {best_rmsle_subset:.4f} @ step {best_step}")

del trainer
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after train cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")

## 5. Final eval — generative RMSLE tren full 3,926 val

Dung best ckpt (`load_best_model_at_end=True` da auto load lai vao `model`).
Chay 1 lan tren full val, luu `results/v4_resume_results.json`.

In [ ]:
model.eval()
for _m in model.modules():
    if isinstance(_m, torch.nn.Conv1d):
        _m.to(torch.bfloat16)

def predict_one(prompt: str) -> tuple:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(PARSE_REGEX, gen)
    if m:
        pk = max(PRED_CLAMP_MIN, min(int(float(m.group())), PRED_CLAMP_MAX))
    else:
        pk = 0
    return pk, gen

val_for_eval = val_full_raw if VAL_FULL_SIZE is None else val_full_raw.select(range(VAL_FULL_SIZE))
preds_vnd, trues_vnd, raw_outs = [], [], []
clamp_count = 0
t_eval_start = time.time()

for item in tqdm(val_for_eval, desc="Generative eval (full val)"):
    pk, raw = predict_one(item["prompt"])
    preds_vnd.append(pk * 1000)
    trues_vnd.append(item["price_vnd_true"])
    raw_outs.append(raw)
    if pk in (PRED_CLAMP_MIN, PRED_CLAMP_MAX):
        clamp_count += 1

t_eval = time.time() - t_eval_start
metrics_final = compute_metrics(
    np.array(trues_vnd, dtype=float), np.array(preds_vnd, dtype=float)
)
n_eval = len(val_for_eval)

print("=" * 50)
print(f"v4-resume — {n_eval} val (best ckpt by eval_rmsle)")
print("=" * 50)
print(f"RMSLE : {metrics_final['rmsle']:.4f}  (primary)")
print(f"MAE   : {metrics_final['mae']:,.0f} VND")
print(f"MAPE  : {metrics_final['mape']:.1f}%")
print(f"R2    : {metrics_final['r2']:.4f}")
print(f"Zero preds   : {preds_vnd.count(0)}")
print(f"Clamp trigger: {clamp_count}")
print(f"Sec/item     : {t_eval / n_eval:.2f}s")
print("=" * 50)
print(f"v3 ref       : RMSLE=0.4426")
print(f"Day4 v8 ref  : RMSLE=0.4004")
print(f"Target Stage1: RMSLE 0.40-0.42")

In [ ]:
# Save predictions full + results JSON
preds_dump = [
    {"idx": i, "pred_vnd": int(preds_vnd[i]), "true_vnd": int(trues_vnd[i])}
    for i in range(n_eval)
]
preds_file = NOTEBOOK_DIR / "results" / "v4_resume_val_predictions.json"
with open(preds_file, "w", encoding="utf-8") as f:
    json.dump(preds_dump, f, ensure_ascii=False)
print(f"Saved predictions: {preds_file}")

samples_out = []
for i in range(min(20, n_eval)):
    tv, pv = trues_vnd[i], preds_vnd[i]
    err_pct = abs(pv - tv) / tv * 100 if tv > 0 else None
    samples_out.append({
        "idx": i,
        "prompt_excerpt": val_for_eval[i]["prompt"][:120],
        "generated_raw": raw_outs[i],
        "pred_vnd": pv,
        "true_vnd": tv,
        "error_pct": round(err_pct, 1) if err_pct is not None else None,
    })

results = {
    "version": "v4_resume",
    "model": BASE_MODEL,
    "dataset": DATASET_NAME,
    "resume_from": RESUME_ADAPTER,
    "config": {
        "train_size": len(train_ds_tok),
        "val_callback_size": VAL_CALLBACK_SIZE,
        "val_full_size": n_eval,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "target_modules": LORA_TARGET_MODULES,
        "num_epochs": NUM_EPOCHS,
        "per_device_batch": PER_DEVICE_BATCH,
        "grad_accum": GRAD_ACCUM,
        "learning_rate": LEARNING_RATE,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
        "neftune_alpha": NEFTUNE_ALPHA,
        "group_by_length": GROUP_BY_LENGTH,
        "gradient_checkpointing": GRADIENT_CHECKPOINTING,
        "max_seq_length": MAX_SEQ_LENGTH,
        "tokens_fixed": TOKENS_FIXED,
        "max_summary_tokens": MAX_SUMMARY_TOKENS,
        "early_stop_patience": EARLY_STOP_PATIENCE,
    },
    "vram_train_peak_gb": round(final_vram_peak, 2),
    "total_train_sec": round(total_train_sec, 1),
    "sec_per_val_item": round(t_eval / n_eval, 2),
    "train_loss_curve": train_losses,
    "eval_loss_curve": eval_losses,
    "eval_rmsle_curve": eval_rmsle_curve,
    "metrics_final": metrics_final,
    "samples_20": samples_out,
}

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Saved: {RESULTS_FILE}")

## 6. Plot — 200 random sample

Eval tren full val o cell tren, plot 200 item random (seed=42) cho truc quan.

In [ ]:
rng = np.random.default_rng(SEED)
plot_idx = rng.choice(n_eval, size=min(PLOT_SIZE, n_eval), replace=False)
trues_plot = np.array([trues_vnd[i] for i in plot_idx], dtype=float)
preds_plot = np.array([preds_vnd[i] for i in plot_idx], dtype=float)
names_plot = [val_for_eval[int(i)]["prompt"][:50] for i in plot_idx]
plot_predictions(
    trues_plot, preds_plot,
    title=f"v4-resume ({PLOT_SIZE} random sample tu {n_eval} val)",
    names=names_plot,
)

## 7. Push HF private

In [ ]:
print(f"Pushing adapter to {HF_REPO_ADAPTER} (private)...")
model.push_to_hub(HF_REPO_ADAPTER, private=True)
tokenizer.push_to_hub(HF_REPO_ADAPTER, private=True)
print(f"Pushed: https://huggingface.co/{HF_REPO_ADAPTER}")

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM final: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")

## Leaderboard + log Run #3

Sau khi chay xong, paste bang sau vao `phase2_execution_log.md` muc **Run #3 v4-resume**:

```
| Version | RMSLE | MAE (VND) | MAPE | R2 |
|---------|-------|-----------|------|----|
| v3 final (ep3) | 0.4426 | 80,100 | 37.6% | 0.6639 |
| v4-resume best | <fill> | <fill> | <fill> | <fill> |
| Day4 v8 ref | 0.4004 | 79,853 | 30.7% | 0.6920 |
```

**Decision tree sau Stage 1:**
- RMSLE < 0.40 → mục P0 đạt; vẫn run Stage 3 v4-scratch để stretch < 0.38.
- 0.40 ≤ RMSLE < 0.42 → bám design, Stage 3 mong RMSLE thêm.
- RMSLE ≥ 0.45 → resume bị stuck ở local min v3, bỏ resume, focus Stage 3 v4-scratch (ghi vào log mục "Can Opus xem xet").